In [4]:
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
from bertopic import BERTopic
from konlpy.tag import Okt
import numpy as np
from umap import UMAP
import re
import datetime
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from scipy.spatial.distance import squareform
try:
    from collections.abc import Mapping
except ImportError:
    from collections import Mapping
    
from gensim.models import Word2Vec
from scipy.cluster import hierarchy as sch
import warnings
warnings.filterwarnings('ignore')
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [6]:
stop_word_file = "../datasets/RawDatasets/txt/new_stopwords.txt"
stop_word_file2 = "../datasets/RawDatasets/txt/stopwords.txt"
#stop_word_file3 = "../datasets/RawDatasets/txt/user_dic.txt"
stop_words1 =  [line.strip() for line in open(stop_word_file, encoding="utf-8").readlines()]
stop_words2 =  [line.strip() for line in open(stop_word_file2, encoding="utf-8").readlines()]
#stop_words3 = [line.strip() for line in open(stop_word_file3, encoding="utf-8").readlines()]
stop_words = stop_words1+stop_words2

In [7]:
class CustomTokenizer:
    def __init__(self, tagger):
        self.tagger = tagger
    def __call__(self, sent):
        #sent = sent[:1000000]
        hangul = re.compile('[^ 0-9가-힣+]')
        sent = hangul.sub(' ', sent)
        sent = " ".join(sent.split())
        word_tokens = self.tagger.pos(sent, stem=True)
        temp = [word[0] for word in word_tokens if (word[1] =='Adjective' or  word[1] =='Noun')]
        result = [word for word in temp if (len(word) > 1  and ( not word in stop_words))]
        #한 단어짜리 토큰 제거
        #불용어 제거
        return result

In [8]:
def get_word_embedding_based_similarity(topic1, topic2):
    similarity = 0
    for i in topic1:
        for j in topic2:
            similarity+= word2vec_model.wv.similarity(i,j)
    return similarity/(len(topic1)*len(topic2))

In [9]:
def get_coherence(mdl,k):
    topics = []
    for i in range(k):
        topics.append([word[0] for word in mdl.get_topic(i)])

    cm = CoherenceModel(topics = topics,
                   #corpus = corpus,
                    texts =texts,
                    dictionary = dictionary,
                   coherence = 'c_npmi')
    return cm.get_coherence()

In [10]:
def get_hierarchy(model, k):
    distance_function = lambda x: 1 - cosine_similarity(x)
    linkage_function = lambda x: sch.linkage(x, 'ward', optimal_ordering=True)

    # Calculate distance
    embeddings = model.c_tf_idf_[model._outliers:k]
    X = distance_function(embeddings)

    # Make sure it is the 1-D condensed distance matrix with zeros on the diagonal
    np.fill_diagonal(X, 0)
    X = squareform(X)

    # Use the 1-D condensed distance matrix as an input instead of the raw distance matrix
    Z = linkage_function(X)

    # Calculate basic bag-of-words to be iteratively merged later
    documents = pd.DataFrame({"Document": docs,
                              "ID": range(len(docs)),
                              "Topic": model.topics_})
    documents_per_topic = documents.groupby(['Topic'], as_index=False).agg({'Document': ' '.join})
    documents_per_topic.loc[documents_per_topic.Topic != -1, :]
    ## top 10 topics
    documents_per_topic = documents_per_topic.loc[:k]
    clean_documents = documents_per_topic.Document.values

    # Scikit-Learn Deprecation: get_feature_names is deprecated in 1.0
    # and will be removed in 1.2. Please use get_feature_names_out instead.
    #if version.parse(sklearn_version) >= version.parse("1.0.0"):
    #words = model.vectorizer_model.get_feature_names_out()
    #else:
    words = model.vectorizer_model.get_feature_names()

    bow = model.vectorizer_model.transform(clean_documents)

    # Extract clusters
    hier_topics = pd.DataFrame(columns=["Parent_ID", "Parent_Name", "Topics",
                                        "Child_Left_ID", "Child_Left_Name",
                                        "Child_Right_ID", "Child_Right_Name"])
    for index in tqdm(range(len(Z))):

        # Find clustered documents
        clusters = sch.fcluster(Z, t=Z[index][2], criterion='distance') - model._outliers
        cluster_df = pd.DataFrame({"Topic": range(len(clusters)), "Cluster": clusters})
        cluster_df = cluster_df.groupby("Cluster").agg({'Topic': lambda x: list(x)}).reset_index()
        #cluster_df = cluster_df.loc[:10]
        nr_clusters = len(clusters)

        # Extract first topic we find to get the s aet of topics in a merged topic
        topic = None
        val = Z[index][0]
        while topic is None:
            if val - len(clusters) < 0:
                topic = int(val)
            else:
                val = Z[int(val - len(clusters))][0]
        clustered_topics = [i for i, x in enumerate(clusters) if x == clusters[topic]]

        # Group bow per cluster, calculate c-TF-IDF and extract words
        grouped = csr_matrix(bow[clustered_topics].sum(axis=0))
        c_tf_idf = model.ctfidf_model.fit_transform(grouped)
        selection = documents.loc[documents.Topic.isin(clustered_topics), :]
        selection.Topic = 0
        words_per_topic = model._extract_words_per_topic(words, selection, c_tf_idf)

        # Extract parent's name and ID
        parent_id = index + len(clusters)
        parent_name = "_".join([x[0] for x in words_per_topic[0]][:10])

        # Extract child's name and ID
        Z_id = Z[index][0]
        child_left_id = Z_id if Z_id - nr_clusters < 0 else Z_id - nr_clusters

        if Z_id - nr_clusters < 0:
            child_left_name = "_".join([x[0] for x in model.get_topic(Z_id)][:10])
        else:
            child_left_name = hier_topics.iloc[int(child_left_id)].Parent_Name

        # Extract child's name and ID
        Z_id = Z[index][1]
        child_right_id = Z_id if Z_id - nr_clusters < 0 else Z_id - nr_clusters

        if Z_id - nr_clusters < 0:
            child_right_name = "_".join([x[0] for x in model.get_topic(Z_id)][:10])
        else:
            child_right_name = hier_topics.iloc[int(child_right_id)].Parent_Name

        # Save results
        hier_topics.loc[len(hier_topics), :] = [parent_id, parent_name,
                                                clustered_topics,
                                                int(Z[index][0]), child_left_name,
                                                int(Z[index][1]), child_right_name]

    hier_topics["Distance"] = Z[:, 2]
    hier_topics = hier_topics.sort_values("Parent_ID", ascending=False)
    hier_topics[["Parent_ID", "Child_Left_ID", "Child_Right_ID"]] = hier_topics[["Parent_ID", "Child_Left_ID", "Child_Right_ID"]].astype(str)
    return hier_topics

In [13]:
data = pd.read_feather('../datasets/covid19_1.ftr')
texts =data.okt
dictionary = Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

In [14]:
word2vec_model = Word2Vec.load('./models/word2vec.model')

In [15]:
from sentence_transformers import SentenceTransformer
#sentence_model = SentenceTransformer("jhgan/ko-sroberta-multitask")
umap_model = UMAP(n_neighbors=15, n_components=5, 
                  min_dist=0.0, metric='cosine', random_state=33)
custom_tokenizer = CustomTokenizer(Okt())
vectorizer = CountVectorizer(tokenizer = custom_tokenizer)


In [16]:
model_list = []
tree_list = []
for i in tqdm([1,2,3,4]):
    docs = pd.read_feather(f'./Datasets/covid{i}.ftr').corrected_twit.to_list()
    model = BERTopic(language = 'Korean',
         top_n_words=20,
         #nr_topics= "auto",
         #embedding_model = sentence_model,
         umap_model = umap_model,
         vectorizer_model = vectorizer,
         ) 
    topic, prob = model.fit_transform(docs)
    #hier_topics = model.hierarchical_topics(docs)
    
    model_list.append(model)
    #tree_list.append(hier_topics)

100%|█████████████████████████████████████████████████████████████████| 4/4 [1:56:40<00:00, 1750.20s/it]


In [17]:
score_df = pd.DataFrame(columns=['phase','k','hcsd'])
for phase in [1,2,3,4]:
    data = pd.read_feather(f'./Datasets/covid{phase}.ftr')
    docs = pd.read_feather(f'./Datasets/covid{phase}.ftr').corrected_twit.to_list()
    texts =data.okt
    dictionary = Dictionary(texts)
    corpus = [dictionary.doc2bow(text) for text in texts]
    
    for topic_num in range(10,55,5):
        hier_topics = get_hierarchy(model_list[phase-1],topic_num)
        hier_topics['words'] = range(len(hier_topics))
        hier_topics['lc_words'] = range(len(hier_topics))
        hier_topics['rc_words'] = range(len(hier_topics))
        for i in range(len(hier_topics)):
            hier_topics['lc_words'][i] = [word for word in hier_topics['Child_Left_Name'][i].split('_') if (word !='')]
            hier_topics['rc_words'][i] = [word for word in hier_topics['Child_Right_Name'][i].split('_') if (word !='')]
            hier_topics['words'][i] = [word for word in hier_topics['Parent_Name'][i].split('_') if (word !='')]

        similarity = 0
        diversity = 0
        for i in range(len(hier_topics)):
            #similarity
            similarity+=get_word_embedding_based_similarity(hier_topics.loc[i].words,hier_topics.loc[i].lc_words)
            similarity+=get_word_embedding_based_similarity(hier_topics.loc[i].words,hier_topics.loc[i].rc_words)
            #diversity

            diversity += (1 - get_word_embedding_based_similarity(hier_topics.loc[i].lc_words,hier_topics.loc[i].rc_words))

            #print("#unique words :", len(unique_words))
            #print("#lc+rc : ",len(lc)+len(rc))

        similarity /=(len(hier_topics)*2)
        diversity /= len(hier_topics)

        coherence = get_coherence(model, len(model.get_topic_info())-1)
        #coherence normalization
        coherence = (coherence + 1)/2
        score = (similarity+diversity+coherence)/3
        score_df = score_df.append({'phase':phase,'k':topic_num,'hcsd':score},ignore_index=True)
        print("phase",phase,"topic number:",topic_num,similarity,diversity,coherence,score)
    print()


100%|███████████████████████████████████████████████████| 8/8 [00:00<00:00, 106.88it/s]


phase 1 topic number: 10 0.44419244418360904 0.53709466067834 0.36367326315746495 0.448320122673138


100%|█████████████████████████████████████████████████| 13/13 [00:00<00:00, 105.20it/s]


phase 1 topic number: 15 0.4401097026016438 0.47844477184712353 0.36367326315746495 0.4274092458687441


100%|██████████████████████████████████████████████████| 18/18 [00:00<00:00, 96.49it/s]


...

phase 4 topic number: 40 0.411093191197142 0.7030465202219784 0.40604476954875285 0.5067281603226245


100%|████████████████████████████████████████████████████| 1/1 [00:00<00:00, 95.40it/s]


phase 4 topic number: 45 0.411093191197142 0.7030465202219784 0.40604476954875285 0.5067281603226245


100%|████████████████████████████████████████████████████| 1/1 [00:00<00:00, 91.75it/s]


phase 4 topic number: 50 0.411093191197142 0.7030465202219784 0.40604476954875285 0.5067281603226245



In [19]:
score_df

,phase,k,hcsd
0,1.0,10.0,0.448320
1,1.0,15.0,0.427409
2,1.0,20.0,0.430154
3,1.0,25.0,0.427550
4,1.0,30.0,0.427597
5,1.0,35.0,0.426126
6,1.0,40.0,0.428505
7,1.0,45.0,0.428028
8,1.0,50.0,0.430358
9,2.0,10.0,0.461459


In [21]:
score_df.to_feather('./result/bertopic_result.ftr')

In [24]:
model_list[1].get_topic_info()

,Topic,Count,Name
0,-1,39553,-1_아프다_주사_괜찮다_근육통
1,0,3120,0_잔여백신_이름_예약_스티커
2,1,2003,1_언니_임도_괜찮다_이재명
3,2,1636,2_타이레놀_복용_미리_부루펜
4,3,1582,3_예약_잔여백신_추석_무섭다
...,...,...,...
305,304,10,304_체온_측정_리치_미팅
306,305,10,305_다라_접속_차라_훈련
307,306,10,306_탱고_자유분방하다_즉흥_짜릿하다
308,307,10,307_로션_보이시_편차_잡담


In [22]:
hier_topics

,Parent_ID,Parent_Name,Topics,Child_Left_ID,Child_Left_Name,Child_Right_ID,Child_Right_Name,Distance,words,lc_words,rc_words
0,2,아프다_혈통_부작용_예약_동절기_괜찮다_추가_주사_병원_증상,"[0, 1]",0,아프다_혈통_부작용_예약_동절기_괜찮다_추가_주사_병원_증상,1,일본_인정_일본도_정치권_트랙_출국_거주_증명서_시즌_한국,0.87395,"[아프다, 혈통, 부작용, 예약, 동절기, 괜찮다, 추가, 주사, 병원, 증상]","[아프다, 혈통, 부작용, 예약, 동절기, 괜찮다, 추가, 주사, 병원, 증상]","[일본, 인정, 일본도, 정치권, 트랙, 출국, 거주, 증명서, 시즌, 한국]"


# Interpretation

In [33]:
tree = model.get_topic_tree(hier_topics)

In [34]:
print(tree)

.
├─혈통_아프다_생리_부작용_예약
│    ├─혈통_아프다_생리_부작용_예약
│    │    ├─생리_부작용_생리통_정부_쓰레기
│    │    │    ├─부작용_식욕_피곤하다_엄청나다_불면증
│    │    │    │    ├─■──식욕_고프다_햄버거_부작용_폭발 ── Topic: 7
│    │    │    │    └─부작용_피곤하다_불면증_종일_수면제
│    │    │    │         ├─■──피곤하다_부작용_불면증_종일_수면제 ── Topic: 4
│    │    │    │         └─■──부작용_혓바늘_자건_자력_스킬 ── Topic: 31
│    │    │    └─생리_생리통_정부_쓰레기_국민
│    │    │         ├─■──생리_생리통_출혈_생리주기_생리전증후군 ── Topic: 3
│    │    │         └─■──정부_국민_쓰레기_미국_기자 ── Topic: 1
│    │    └─혈통_아프다_예약_완료_추가
│    │         ├─혈통_아프다_예약_완료_추가
│    │         │    ├─■──졸지_아프다_날씨_혈통_달성 ── Topic: 28
│    │         │    └─혈통_아프다_예약_완료_추가
│    │         │         ├─■──혈통_예약_완료_추가_아프다 ── Topic: 0
│    │         │         └─■──멀쩡하다_아프다_괜찮다_증상_다행 ── Topic: 2
│    │         └─■──차갑다_아프다_힘들다_후유증_가다실 ── Topic: 5
│    └─엄마_스타일_알레르기_언니_몸살
│         ├─■──엄마_알레르기_언니_가족_미화 ── Topic: 10
│         └─■──엄마_스타일_서도_독하다_두드러기 ── Topic: 13
└─겨드랑이_탈모_뽀로로_부루펜_비아그라
     ├─겨드랑이_탈모_뽀로로_부루펜_비아그라
     │    ├─부루펜_타이레놀_미리_괜찮다_금방

In [35]:
hier_topics

,Parent_ID,Parent_Name,Topics,Child_Left_ID,Child_Left_Name,Child_Right_ID,Child_Right_Name,Distance,words,lc_words,rc_words
31,64,혈통_아프다_부작용_생리_예약,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",62,혈통_아프다_생리_부작용_예약,63,겨드랑이_탈모_뽀로로_부루펜_비아그라,1.310869,"[혈통, 아프다, 부작용, 생리, 예약]","[혈통, 아프다, 생리, 부작용, 예약]","[겨드랑이, 탈모, 뽀로로, 부루펜, 비아그라]"
30,63,겨드랑이_탈모_뽀로로_부루펜_비아그라,"[6, 8, 9, 11, 12, 14, 15, 16, 17, 18, 19, 20, ...",60,겨드랑이_탈모_뽀로로_부루펜_비아그라,35,무섭다_무서움_폭력_처형_무경,1.271215,"[겨드랑이, 탈모, 뽀로로, 부루펜, 비아그라]","[겨드랑이, 탈모, 뽀로로, 부루펜, 비아그라]","[무섭다, 무서움, 폭력, 처형, 무경]"
29,62,혈통_아프다_생리_부작용_예약,"[0, 1, 2, 3, 4, 5, 7, 10, 13, 28, 31]",61,혈통_아프다_생리_부작용_예약,33,엄마_스타일_알레르기_언니_몸살,1.228505,"[혈통, 아프다, 생리, 부작용, 예약]","[혈통, 아프다, 생리, 부작용, 예약]","[엄마, 스타일, 알레르기, 언니, 몸살]"
28,61,혈통_아프다_생리_부작용_예약,"[0, 1, 2, 3, 4, 5, 7, 28, 31]",49,생리_부작용_생리통_정부_쓰레기,38,혈통_아프다_예약_완료_추가,1.139165,"[혈통, 아프다, 생리, 부작용, 예약]","[생리, 부작용, 생리통, 정부, 쓰레기]","[혈통, 아프다, 예약, 완료, 추가]"
27,60,겨드랑이_탈모_뽀로로_부루펜_비아그라,"[6, 8, 9, 11, 12, 14, 15, 16, 17, 18, 19, 20, ...",39,부루펜_타이레놀_미리_괜찮다_금방,59,겨드랑이_탈모_뽀로로_비아그라_이재명,1.109596,"[겨드랑이, 탈모, 뽀로로, 부루펜, 비아그라]","[부루펜, 타이레놀, 미리, 괜찮다, 금방]","[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]"
26,59,겨드랑이_탈모_뽀로로_비아그라_이재명,"[6, 8, 9, 12, 14, 15, 16, 17, 18, 19, 20, 22, ...",58,겨드랑이_탈모_뽀로로_비아그라_이재명,40,추가_남편_가족_아빠_킨님,1.073156,"[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]","[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]","[추가, 남편, 가족, 아빠, 킨님]"
25,58,겨드랑이_탈모_뽀로로_비아그라_이재명,"[6, 8, 9, 12, 14, 15, 16, 19, 20, 22, 24, 25, ...",48,고맙다_무덥다_외치_지화_축하,57,겨드랑이_탈모_뽀로로_비아그라_이재명,1.069516,"[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]","[고맙다, 무덥다, 외치, 지화, 축하]","[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]"
24,57,겨드랑이_탈모_뽀로로_비아그라_이재명,"[8, 12, 14, 16, 19, 20, 22, 24, 25, 27, 29, 30...",56,겨드랑이_탈모_뽀로로_비아그라_이재명,45,커피_진정하다_카페인_심장_위반,1.045999,"[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]","[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]","[커피, 진정하다, 카페인, 심장, 위반]"
23,56,겨드랑이_탈모_뽀로로_비아그라_이재명,"[8, 12, 14, 16, 19, 20, 22, 25, 27, 30, 32]",44,비아그라_회사_묵음_스펠링_이름,55,겨드랑이_탈모_뽀로로_이재명_밴드,1.039266,"[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]","[비아그라, 회사, 묵음, 스펠링, 이름]","[겨드랑이, 탈모, 뽀로로, 이재명, 밴드]"
22,55,겨드랑이_탈모_뽀로로_이재명_밴드,"[8, 12, 14, 16, 19, 20, 25, 27, 32]",54,겨드랑이_뽀로로_밴드_조종_삼겹살,46,탈모_이재명_상담_부작용_죄명,1.029846,"[겨드랑이, 탈모, 뽀로로, 이재명, 밴드]","[겨드랑이, 뽀로로, 밴드, 조종, 삼겹살]","[탈모, 이재명, 상담, 부작용, 죄명]"
